# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivanilokh/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will use a Random Forest classifier to identify whether a content page is declining. Random Forest is suitable because it can capture non-linear relationships between content and performance features and can provide feature importance for interpretation. I will compare it with the Week-4 rule-based baseline using the same client-level holdout split and Precision@50.

In [7]:
!git clone https://github.com/shivanilokh/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [8]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Declining pages:", df["is_declining_label"].sum())

Rows: 30000
Columns: 45
Declining pages: 16262


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a client-level holdout split. Approximately 20% of clients will be held out for testing, so pages from the same client do not appear in both training and test data. This is more honest than a random row split because it tests whether the model generalizes to unseen clients.

In [10]:
from sklearn.model_selection import train_test_split

clients = df["client_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df = df[df["client_id"].isin(test_clients)].copy()

print("Total clients:", len(clients))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print(
    "Client overlap:",
    len(set(train_df["client_id"]) & set(test_df["client_id"]))
)

Total clients: 32
Train clients: 25
Test clients: 7
Train rows: 26581
Test rows: 3419
Client overlap: 0


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Training and comparison

The target is `is_declining_label`, where 1 means the page is declining and 0 means it is not. I will exclude `trend_direction` and `trend_pct` because they are used to construct the target and would cause target leakage. I will train the Random Forest on the client-level training set and evaluate the ranked predictions on the held-out clients using Precision@50.

In [12]:
# Target
target = "is_declining_label"

# Columns that must not be used as model features
exclude_cols = {
    target,
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
}

feature_cols = [c for c in df.columns if c not in exclude_cols]

X_train = train_df[feature_cols].copy()
y_train = train_df[target].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target].copy()

# Separate numeric and categorical features
numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf)
    ]
)

# Train
model.fit(X_train, y_train)

# Probability of decline
test_proba = model.predict_proba(X_test)[:, 1]

# Rank pages by probability
test_results = test_df[
    ["content_id", "client_id", target]
].copy()

test_results["decline_probability"] = test_proba

test_results = test_results.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

# Precision@50
top_50 = test_results.head(50)

model_precision_at_50 = top_50[target].mean()

print("Random Forest Precision@50:", round(model_precision_at_50, 3))

# Week-4 baseline score on the SAME test rows
baseline_test = test_df.copy()

baseline_test["baseline_score"] = (
    (baseline_test["days_since_last_update"] >= 180).astype(int) * 3
    + (baseline_test["impressions_90d"] >= 5000).astype(int) * 2
    + (baseline_test["ctr"] < 0.5).astype(int)
)

baseline_test = baseline_test.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

baseline_top_50 = baseline_test.head(50)

baseline_precision_at_50 = baseline_top_50[target].mean()

print("Week-4 Baseline Precision@50:", round(baseline_precision_at_50, 3))

# Comparison table
comparison = pd.DataFrame({
    "Model": [
        "Week-4 Rule-Based Baseline",
        "Week-5 Random Forest"
    ],
    "Precision@50": [
        baseline_precision_at_50,
        model_precision_at_50
    ]
})

comparison

Random Forest Precision@50: 1.0
Week-4 Baseline Precision@50: 0.54


,Model,Precision@50
0,Week-4 Rule-Based Baseline,0.54
1,Week-5 Random Forest,1.00


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Error analysis

I reviewed the highest-ranked predictions and the cases where the model's ranking differs from the baseline. The model is designed to prioritize pages with a higher estimated probability of decline. Some errors are expected because declining performance can be influenced by several interacting content and search-performance signals. The model should therefore be treated as decision-support rather than an automatic refresh decision.

In [14]:
# Top 10 model predictions
print("Top 10 model-ranked pages:")
display(
    test_results[
        [
            "content_id",
            "client_id",
            "decline_probability",
            target
        ]
    ].head(10)
)

# False positives / false negatives using 0.5 threshold
test_results["predicted_label"] = (
    test_results["decline_probability"] >= 0.5
).astype(int)

false_positives = test_results[
    (test_results["predicted_label"] == 1)
    & (test_results[target] == 0)
]

false_negatives = test_results[
    (test_results["predicted_label"] == 0)
    & (test_results[target] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

# Feature importance
fitted_preprocessor = model.named_steps["preprocessor"]
fitted_rf = model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": fitted_rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Top 10 features:")
display(feature_importance.head(10))

Top 10 model-ranked pages:


,content_id,client_id,decline_probability,is_declining_label
0,content_c92cbdb448d0,client_bbb965ab0c,0.983333,1
1,content_6752b50ee577,client_bbb965ab0c,0.976667,1
2,content_ca122cc888aa,client_bbb965ab0c,0.976667,1
3,content_7ba9813c1beb,client_a88a7902cb,0.973333,1
4,content_78d59b5950e9,client_bbb965ab0c,0.973333,1
5,content_870430512868,client_a88a7902cb,0.973333,1
6,content_b73061588e7d,client_a88a7902cb,0.973333,1
7,content_5ad0d416fbbd,client_bbb965ab0c,0.970000,1
8,content_29884c0f9255,client_8527a891e2,0.970000,1
9,content_9de926fa1505,client_bbb965ab0c,0.970000,1


False positives: 369
False negatives: 26
Top 10 features:


,feature,importance
18,num__impressions_prev_30d,0.155444
15,num__impressions_last_30d,0.129258
5,num__impressions_90d,0.068868
25,num__avg_position,0.050830
13,num__days_with_impressions,0.047643
21,num__content_age_days,0.040168
17,num__sessions_last_30d,0.028223
3,num__word_count,0.027722
4,num__char_count,0.027179
24,num__ctr,0.025291


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.